In [31]:
import os
import os.path as osp
import json
import pandas as pd
import numpy as np
from typing import List
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    f1_score,
    log_loss,
    brier_score_loss,
    precision_recall_curve,
    auc
)
import xgboost as xgb

In [2]:
try:
    PROJECT_ROOT = osp.abspath(osp.join(osp.dirname(__file__),'..'))
except:
    PROJECT_ROOT = osp.abspath(osp.join(os.getcwd(),'..'))

print(PROJECT_ROOT)
STATIONS_FILE = osp.join(PROJECT_ROOT,'src','stations_infos.json')

with open(STATIONS_FILE) as src:
        STATIONS_INFOS = json.load(src)

/home/bfeldmann/projects/formation_mlops/projet_meteoAUS


In [38]:
# =========================================================
# 🔧 FEATURE ENGINEERING AUTOMATIQUE
# =========================================================
class CreateFeatures:
    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.dict_bool = {'No': 0, 'Yes': 1}
        self.direction_to_angle = {
            "N": 0.0, "NNE": 22.5, "NE": 45.0, "ENE": 67.5,
            "E": 90.0, "ESE": 112.5, "SE": 135.0, "SSE": 157.5,
            "S": 180.0, "SSW": 202.5, "SW": 225.0, "WSW": 247.5,
            "W": 270.0, "WNW": 292.5, "NW": 315.0, "NNW": 337.5
        }
    
    def build_features(self):
        date_col = 'Date'
        location_col = 'Location' 
        list_dir_cols = ['WindGustDir']
        list_lag_cols = ['WindGustDir_sin', 'WindGustDir_cos', 'MinTemp', 'MaxTemp', 'Rainfall',
                         'WindGustSpeed', 'Humidity9am', 'Pressure9am']

        self.df = self.df.sort_values(date_col)

        self.df['RainToday'] = self.df['RainToday'].apply(lambda x: self.dict_bool.get(x,np.nan))
        self.df['RainTomorrow'] = self.df['RainTomorrow'].apply(lambda x: self.dict_bool.get(x,np.nan))

        self.create_time_features(date_col)
        self.create_coords_features(location_col)
        self.create_sin_cos_features(list_dir_cols)
        self.create_lag_features(list_lag_cols, lags=[1,3])
        # self.create_rolling_features(self, cols: List[str], windows: List[int]):

        list_cols_drop = [date_col, location_col] + list_dir_cols + \
            ['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm', 'WindDir9am', 'WindDir3pm']
        self.df.drop(list_cols_drop, inplace=True, axis=1)

    def create_time_features(self, date_col: str):
        self.df["dayofyear"] = self.df[date_col].dt.dayofyear

    def create_coords_features(self, location_col: str, latlong=None):
        if latlong is not None:
            self.df['latitude'] = latlong[0]
            self.df['longitude'] = latlong[1]
        else:
            self.df['latitude'] = self.df[location_col].apply(lambda x: STATIONS_INFOS.get(x, {'latlong':[0,0]})['latlong'][0])
            self.df['longitude'] = self.df[location_col].apply(lambda x: STATIONS_INFOS.get(x, {'latlong':[0,0]})['latlong'][1])
            self.df.loc[(self.df["latitude"] == 0) & (self.df["longitude"] == 0), ['latitude', 'longitude']] = np.nan
        
    def create_sin_cos_features(self, cols: List[str]):
        for col in cols:
            self.df[f'{col}_sin'] = self.df[col].apply(
                lambda x: np.sin(np.deg2rad(self.direction_to_angle.get(x,0)))
            )
            self.df[f'{col}_cos'] = self.df[col].apply(
                lambda x: np.cos(np.deg2rad(self.direction_to_angle.get(x,0)))
            )

    def create_lag_features(self, cols: List[str], lags: List[int]):
        for col in cols:
            for lag in lags:
                self.df[f"{col}_lag_{lag}"] = self.df[col].shift(lag)

    def create_rolling_features(self, cols: List[str], windows: List[int]):
        for col in cols:
            for window in windows:
                self.df[f"{col}_roll_mean_{window}"] = np.nanmean(self.df[col].rolling(window))
                self.df[f"{col}_roll_std_{window}"] = np.nanstd(self.df[col].rolling(window))

In [48]:
# =========================================================
# 📊 METRICS
# =========================================================

def evaluate(model, X, y, name="set", threshold=None):
    proba = model.predict_proba(X)[:, 1]

    if threshold is None:
        best_threshold, best_f1_score = optimize_threshold(y,proba)
    else:
        best_threshold = threshold
        best_f1_score = None

    pred = (proba >= best_threshold).astype(int)

    # métriques classiques
    acc = accuracy_score(y, pred)
    roc_auc = roc_auc_score(y, proba)
    recall = recall_score(y, pred)
    f1 = f1_score(y, pred)
    logloss = log_loss(y, proba)
    brier = brier_score_loss(y, proba)

    # PR AUC
    precision, recall_curve, _ = precision_recall_curve(y, proba)
    pr_auc = auc(recall_curve, precision)

    print(f"\n📊 {name}")
    print(f"Best Threshold : {best_threshold:.3f}")
    if best_f1_score is not None:
        print(f"Best F1-score : {best_f1_score:.3f}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"ROC AUC   : {roc_auc:.4f}")
    print(f"PR AUC    : {pr_auc:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")
    print(f"LogLoss   : {logloss:.4f}")
    print(f"Brier     : {brier:.4f}")

    return best_threshold

def optimize_threshold(y_true, proba):
    best_t = 0.5
    best_f1 = 0

    for t in np.linspace(0.3, 0.8, 100):
        pred = (proba >= t).astype(int)
        f1 = f1_score(y_true, pred)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

# =========================================================
# 🚀 PIPELINE PRINCIPAL
# =========================================================

def train_xgboost_pipeline(dataset_train, dataset_validation, dataset_test, target_col="RainTomorrow"):
    train_df = dataset_train.copy()
    val_df   = dataset_validation.copy()
    test_df  = dataset_test.copy()

    # --- séparation target
    y_train = train_df[target_col].astype(int)
    y_val   = val_df[target_col].astype(int)
    y_test  = test_df[target_col].astype(int)

    X_train = train_df.drop(columns=target_col)
    X_val   = val_df.drop(columns=target_col)
    X_test  = test_df.drop(columns=target_col)

    # =====================================================
    # 🌍 ENCODING CITY (robuste et cohérent)
    # =====================================================

    # if "city" in X_train.columns:

    #     all_cities = pd.concat([
    #         X_train["city"],
    #         X_val["city"],
    #         X_test["city"]
    #     ]).astype("category")

    #     categories = all_cities.cat.categories

    #     def encode(df):
    #         df = df.copy()
    #         df["city"] = pd.Categorical(df["city"], categories=categories).codes
    #         return df

    #     X_train = encode(X_train)
    #     X_val   = encode(X_val)
    #     X_test  = encode(X_test)

    # =====================================================
    # 🤖 MODEL
    # =====================================================

    model = xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=15,
        reg_alpha=0.3,
        reg_lambda=1.0,
        eval_metric="logloss",
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # =====================================================
    # 📊 EVALUATION
    # =====================================================

    threshold_validation = evaluate(model, X_val, y_val, "VALIDATION")
    _ = evaluate(model, X_train, y_train, "TRAIN", threshold=threshold_validation)
    _ = evaluate(model, X_test, y_test, "TEST", threshold=threshold_validation)

    return model

In [4]:
dict_cols = {
    'Date':'datetime','Location': 'str', 'MinTemp': 'float32', 'MaxTemp': 'float32', 'Rainfall': 'float32', 'Evaporation': 'float32',
    'Sunshine': 'float32', 'WindGustDir': 'string', 'WindGustSpeed': 'float32',
    'Temp9am': 'float32', 'Humidity9am': 'int16', 'Cloud9am': 'float32', 'WindDir9am': 'string',
    'WindSpeed9am': 'float32', 'Pressure9am': 'float32', 'Temp3pm': 'float32', 'Humidity3pm': 'int16',
    'Cloud3pm': 'float32', 'WindDir3pm': 'string', 'WindSpeed3pm': 'float32', 'Pressure3pm': 'float32',
    'RainToday': 'str', 'RainTomorrow': 'str'
    }

In [34]:
filepath = osp.join(PROJECT_ROOT,'data','weatherAUS.csv')
df = pd.read_csv(filepath)

df = df.replace(
    [pd.NA, None, "NaN", "nan", "NAN", "NA", "N/A", "", " ", "null", "None"],
    np.nan
)
# print(df.dtypes)
# df.head()

for col,dtype in dict_cols.items():
    if dtype in ('int16', 'float32'):
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype(dtype)
        df[col] = df[col].replace(-1, np.nan)
        
    elif dtype=='datetime':
        df[col] = pd.to_datetime(df[col], errors='coerce')
        
    else:
        df[col] = df[col].astype(dtype)

df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.900000,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.700012,1007.099976,8.0,NaN,16.900000,21.799999,No,No
1,2008-12-02,Albury,7.4,25.100000,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.599976,1007.799988,NaN,NaN,17.200001,24.299999,No,No
2,2008-12-03,Albury,12.9,25.700001,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.599976,1008.700012,NaN,2.0,21.000000,23.200001,No,No
3,2008-12-04,Albury,9.2,28.000000,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.599976,1012.799988,NaN,NaN,18.100000,26.500000,No,No
4,2008-12-05,Albury,17.5,32.299999,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.799988,1006.000000,7.0,8.0,17.799999,29.700001,No,No


In [39]:
list_city  = np.unique(df['Location'])
split_train_test = 0.8

# dataset validation
city_validation = 'MelbourneAirport'
extract_df = df[df['Location']==city_validation]

features = CreateFeatures(extract_df)
features.build_features()
data_valid_df = features.df.dropna()


list_train_df = []
list_test_df = []
for city in list_city:
    if city == city_validation:
        continue

    extract_df = df[df['Location']==city]

    features = CreateFeatures(extract_df)
    features.build_features()
    new_df = features.df.dropna()
    print(f'{city} : {len(extract_df)} -> {len(new_df)}')
    
    if len(new_df) > 0:
        split = int(len(new_df) * split_train_test)

        X_train, X_test = new_df.iloc[:split], new_df.iloc[split:]
        list_train_df.append(X_train)
        list_test_df.append(X_test)

data_train_df = pd.concat(list_train_df, ignore_index=True)
data_test_df = pd.concat(list_test_df, ignore_index=True)
print(len(data_train_df))
print(len(data_test_df))
print(len(data_valid_df))

Adelaide : 3193 -> 2783
Albany : 3040 -> 0
Albury : 3040 -> 2884
AliceSprings : 3040 -> 2861
BadgerysCreek : 3009 -> 2556
Ballarat : 3040 -> 2810
Bendigo : 3040 -> 2843
Brisbane : 3193 -> 3001
Cairns : 3040 -> 2869
Canberra : 3436 -> 2724
Cobar : 3009 -> 2725
CoffsHarbour : 3009 -> 2443
Dartmoor : 3009 -> 2727
Darwin : 3193 -> 3071
GoldCoast : 3040 -> 2690
Hobart : 3193 -> 3077
Katherine : 1578 -> 617
Launceston : 3040 -> 1776
Melbourne : 3193 -> 1894
Mildura : 3009 -> 2959
Moree : 3009 -> 2460
MountGambier : 3040 -> 2863
MountGinini : 3040 -> 0
Newcastle : 3039 -> 0
Nhil : 1578 -> 1543
NorahHead : 3004 -> 2753
NorfolkIsland : 3009 -> 2764
Nuriootpa : 3009 -> 2821
PearceRAAF : 3009 -> 2038
Penrith : 3039 -> 0
Perth : 3193 -> 3148
PerthAirport : 3009 -> 2879
Portland : 3009 -> 2710
Richmond : 3009 -> 2737
Sale : 3009 -> 2630
SalmonGums : 3001 -> 0
Sydney : 3344 -> 2141
SydneyAirport : 3009 -> 2823
Townsville : 3040 -> 2948
Tuggeranong : 3039 -> 2761
Uluru : 1578 -> 1427
WaggaWagga : 300

In [49]:
model = train_xgboost_pipeline(data_train_df,data_valid_df, data_test_df)


📊 VALIDATION
Best Threshold : 0.593
Best F1-score : 0.678
Accuracy  : 0.8535
ROC AUC   : 0.8888
PR AUC    : 0.7273
Recall    : 0.7121
F1-score  : 0.6781
LogLoss   : 0.3867
Brier     : 0.1229

📊 TRAIN
Best Threshold : 0.593
Accuracy  : 0.9048
ROC AUC   : 0.9601
PR AUC    : 0.8798
Recall    : 0.8396
F1-score  : 0.7935
LogLoss   : 0.2823
Brier     : 0.0850

📊 TEST
Best Threshold : 0.593
Accuracy  : 0.8472
ROC AUC   : 0.8954
PR AUC    : 0.7528
Recall    : 0.7051
F1-score  : 0.6708
LogLoss   : 0.3798
Brier     : 0.1209


In [14]:
for col in features.df.columns:
    if features.df[col].dtype=='str':
        continue
    
    try:
        a = np.isnan(features.df[col].values)
        pourcent_nan = np.count_nonzero(a)/len(a)*100
        print(f'{col} : {pourcent_nan:.2f}')
    except:
        continue


Date : 0.00
MinTemp : 0.49
MaxTemp : 0.09
Rainfall : 0.52
Evaporation : 46.68
Sunshine : 55.73
WindGustSpeed : 9.81
WindSpeed9am : 6.66
WindSpeed3pm : 6.49
Humidity9am : 1.83
Humidity3pm : 0.35
Pressure9am : 6.58
Pressure3pm : 6.43
Cloud9am : 31.20
Cloud3pm : 36.76
Temp9am : 0.55
Temp3pm : 0.20
RainToday : 0.52
RainTomorrow : 0.52
dayofyear : 0.00
latitude : 0.00
WindGustDir_sin : 0.00
WindGustDir_cos : 0.00
WindDir9am_sin : 0.00
WindDir9am_cos : 0.00
WindDir3pm_sin : 0.00
WindDir3pm_cos : 0.00
WindGustDir_sin_lag_1 : 0.03
WindGustDir_sin_lag_3 : 0.09
MinTemp_lag_1 : 0.52
MinTemp_lag_3 : 0.58
MaxTemp_lag_1 : 0.12
MaxTemp_lag_3 : 0.17
Rainfall_lag_1 : 0.55
Rainfall_lag_3 : 0.61
Sunshine_lag_1 : 55.73
Sunshine_lag_3 : 55.73
Evaporation_lag_1 : 46.68
Evaporation_lag_3 : 46.68
WindGustSpeed_lag_1 : 9.84
WindGustSpeed_lag_3 : 9.90
Humidity9am_lag_1 : 1.86
Humidity9am_lag_3 : 1.92
Cloud9am_lag_1 : 31.20
Cloud9am_lag_3 : 31.23
Pressure9am_lag_1 : 6.61
Pressure9am_lag_3 : 6.66


In [ ]:
# =========================================================
# 🚀 EXECUTION
# =========================================================

if __name__ == "__main__":
    df = pd.read_csv("weather.csv", parse_dates=["date"])
    
    model, features = train_xgboost_pipeline(df)

    print("\nFeatures utilisées :")
    print(features.tolist())